## Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [27]:
import os
from langchain.chat_models import init_chat_model
model = init_chat_model(
    "llama-3.1-8b-instant",
    model_provider="groq"
)

print(model)

output_version=None profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True} client=<groq.resources.chat.completions.Completions object at 0x00000128AA85B560> async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000128AA84D850> model_name='llama-3.1-8b-instant' model_kwargs={} groq_api_key=SecretStr('**********') groq_api_base=None groq_proxy=None


In [28]:
response = model.invoke("Explain Newton's third Law")
response

AIMessage(content='Newton\'s third law of motion is a fundamental concept in physics that states:\n\n**"For every action, there is an equal and opposite reaction."**\n\nIn other words, when an object exerts a force on another object, the second object will exert an equal and opposite force on the first object. This law is often represented by the equation:\n\n**F₁ = -F₂ (or F₁ = F₂ in magnitude)**\n\nWhere F₁ is the force exerted by object 1 on object 2, and F₂ is the force exerted by object 2 on object 1.\n\nHere are some examples to illustrate this concept:\n\n1. **Walking**: When you push against the ground with your feet, the ground exerts an equal and opposite force on you, propelling you forward.\n2. **Throwing a ball**: When you throw a ball, the ball exerts a force on your hand, and your hand exerts an equal and opposite force on the ball, propelling it forward.\n3. **Rocket propulsion**: A rocket works by expelling hot gases out of its back, creating a forward force. The rocke

In [32]:
## Tools
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the Weather at a location"""
    return f"It's sunny in {location}"

model_with_tools=model.bind_tools([get_weather])


In [33]:
response=model_with_tools.invoke("Whats the weather in Banglore")
print(response)

content='' additional_kwargs={'tool_calls': [{'id': '2ntbhxp1t', 'function': {'arguments': '{"location":"Banglore"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 218, 'total_tokens': 233, 'completion_time': 0.026321536, 'completion_tokens_details': None, 'prompt_time': 0.073508807, 'prompt_tokens_details': None, 'queue_time': 0.066320812, 'total_time': 0.099830343}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019e3c44-3900-78d1-accf-d9ab7204920c-0' tool_calls=[{'name': 'get_weather', 'args': {'location': 'Banglore'}, 'id': '2ntbhxp1t', 'type': 'tool_call'}] invalid_tool_calls=[] usage_metadata={'input_tokens': 218, 'output_tokens': 15, 'total_tokens': 233}


In [35]:
response.tool_calls

[{'name': 'get_weather',
  'args': {'location': 'Banglore'},
  'id': '2ntbhxp1t',
  'type': 'tool_call'}]

### Tool Execution Loop

In [36]:
#Step1: Model generates tool calls
message= [{"role": "user", "content": "What's the weather in Banglore?"}]
ai_msg = model_with_tools.invoke(message)
message.append(ai_msg)

#Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    #Execute the tool with the generated arguments
    tool_res = get_weather.invoke(tool_call)
    message.append(tool_res)

# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(message)
print(final_response.text)

However, the actual response from the function call should be more detailed. I will provide a more detailed response in the following format.

The function 'get_weather' returns the weather in the specified location.


In [ ]:
message

[{'role': 'user', 'content': "What's the weather in Banglore?"},
 AIMessage(content='', additional_kwargs={'tool_calls': [{'id': 'kekw1pjbt', 'function': {'arguments': '{"location":"Banglore"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 220, 'total_tokens': 235, 'completion_time': 0.026043371, 'completion_tokens_details': None, 'prompt_time': 0.014675882, 'prompt_tokens_details': None, 'queue_time': 0.06049947, 'total_time': 0.040719253}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--019e3c44-5468-7e01-99ec-b4135731d98a-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Banglore'}, 'id': 'kekw1pjbt', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 220, 'output_tokens': 15, 'total_tokens': 235}),
 ToolMessage(content="It